In [100]:
import pandas as pd
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import plotly.express as px


In [101]:
try:
    df=pd.read_csv("concrete_data.csv")
    print('Dataset Loaded Successfully')
except:
    print('Dataset not Found')

Dataset Loaded Successfully


In [102]:
df

,cement,blast_furnace_slag,fly_ash,water,superplasticizer,coarse_aggregate,fine_aggregate,age,concrete_compressive_strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30
...,...,...,...,...,...,...,...,...,...
1025,276.4,116.0,90.3,179.6,8.9,870.1,768.3,28,44.28
1026,322.2,0.0,115.6,196.0,10.4,817.9,813.4,28,31.18
1027,148.5,139.4,108.6,192.7,6.1,892.4,780.0,28,23.70
1028,159.1,186.7,0.0,175.6,11.3,989.6,788.9,28,32.77


In [103]:
df.rename(columns={
    'blast_furnace_slag': 'slag',
    'fly_ash': 'flyash',
    'superplasticizer': 'plasticizer',
    'coarse_aggregate': 'coarse',
    'fine_aggregate': 'fine',
    'concrete_compressive_strength': 'concrete_strength'
}, inplace=True)

In [104]:
df.columns=df.columns.str.strip()

In [105]:
df.isnull().sum().sort_values(ascending=False)

cement               0
slag                 0
flyash               0
water                0
plasticizer          0
coarse               0
fine_aggregate       0
age                  0
concrete_strength    0
dtype: int64

In [106]:
df.columns

Index(['cement', 'slag', 'flyash', 'water', 'plasticizer', 'coarse',
       'fine_aggregate', 'age', 'concrete_strength'],
      dtype='object')

In [107]:
px.violin(df,x='coarse')

In [108]:
px.violin(df,x='concrete_strength')

In [109]:
px.violin(df,x='flyash')

In [110]:
px.box(df,x='cement')

In [111]:
px.box(df,x='slag')

In [112]:
X=df.drop('concrete_strength',axis=1)
y=df['concrete_strength']

In [113]:
from sklearn.model_selection import train_test_split,GridSearchCV,RandomizedSearchCV
from sklearn.compose import ColumnTransformer,make_column_selector as selector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [114]:
num=Pipeline([
    ('scaler',StandardScaler())
])
preprocessor=ColumnTransformer([
    ('nums',num,selector(dtype_include=[np.number]))
])
preprocessor

,transformers,"[('nums', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


In [115]:
from xgboost import XGBRegressor

In [116]:
model=Pipeline([
    ('preprocessing',preprocessor),
    ('model',XGBRegressor())

])
model

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('nums', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [117]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [120]:
param_grid={
    "model__n_estimators":[500],
    'model__max_depth':[10],
    'model__learning_rate':[0.3]
}
cv=GridSearchCV(
    param_grid=param_grid,
    estimator=model,
    cv=5,
    n_jobs=3,
    scoring='neg_mean_squared_error',
    verbose=1,
    error_score='raise'
    

)
cv.fit(X_train,y_train)
y_pred=cv.predict(X_test)

Fitting 5 folds for each of 1 candidates, totalling 5 fits


In [123]:
from sklearn.metrics import root_mean_squared_error,mean_absolute_error,mean_absolute_percentage_error,mean_squared_error

print('RMSE',root_mean_squared_error(y_test,y_pred))
print('MSE',mean_squared_error(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))
print('MAPE',mean_absolute_percentage_error(y_test,y_pred))

RMSE 5.723829896962781
MSE 32.76222868936495
MAE 3.6481872895620406
MAPE 0.11452868893114085
